# RAG v2 — Alert-Triggered Grounded Explanations

Rebuilds §4.4–4.10 of the revised paper, in the style of the original `rag.ipynb`.
v2 changes walked through below: corpus expansion (wearable-relevant guidelines),
corrected query semantics (subject-baseline reference), raw-text citation audit,
judge validation on a corruption benchmark, and the labeled-event evaluation.

# === Imports ===

### === Core ===

In [1]:
import difflib, json, re, sys, time
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd

### === PDF parsing ===

In [2]:
import fitz

### === Embeddings + vector store ===

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

C:\Users\tanvi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### === Local LLM (Ollama) ===

In [4]:
import ollama

### === Detection-side helpers (from the detection notebook's caches) ===

In [5]:
import wfdb
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import ast

# === Paths + constants ===

### === Project root & corpus paths ===

In [6]:
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR   = PROJECT_ROOT / "outputs_v2"
TIER1_DIR    = PROJECT_ROOT / "Dataset" / "RAG corpus (medical literatureguidelines)"
TIER1_V2_DIR = PROJECT_ROOT / "Dataset" / "Tier1_v2"
TIER2_DIR    = PROJECT_ROOT / "Dataset" / "Tier2_literature"
CHROMA_DIR   = PROJECT_ROOT / "chroma_db_v2"
GEN_JSONL    = OUTPUT_DIR / "generation_v2.jsonl"


### === Config ===

In [7]:
EMBED_MODEL  = "all-MiniLM-L6-v2"
LLM_MODEL    = "qwen3.5:9b"
ALT_MODEL    = "llama3.1:8b"
JUDGE_CANDIDATES = ["llama3.1:8b", "gemma4:e4b"]   # gpt-oss excluded per author decision
CHUNK_WORDS, OVERLAP_WORDS, TOP_K, POOL = 500, 50, 5, 20
CHANNELS = ["ecg","resp","bvp","wrist_eda","wrist_temp"]

### === Verify the paths ===

In [8]:
print("Tier-1 v1 PDFs:", len(list(TIER1_DIR.glob('*.pdf'))))
print("Tier-1 v2 texts:", list(TIER1_V2_DIR.glob('*.txt')))
print("Tier-2 buckets:", sorted(d.name for d in TIER2_DIR.iterdir() if d.is_dir()))
print("generation jsonl exists:", GEN_JSONL.exists())

Tier-1 v1 PDFs: 4
Tier-1 v2 texts: [WindowsPath('D:/All_Folder/data project/new_papers/Retrieval-augmented generation for continuous anomaly alerts/Dataset/Tier1_v2/accaha2023_af_guideline.txt'), WindowsPath('D:/All_Folder/data project/new_papers/Retrieval-augmented generation for continuous anomaly alerts/Dataset/Tier1_v2/ehra2022_digital_devices_arrhythmias.txt')]
Tier-2 buckets: ['01_ppg_arrhythmia', '02_ppg_signal_quality', '03_ecg_anomaly_ml', '04_wearable_stress', '05_continuous_monitoring', '06_biosignal_methods']
generation jsonl exists: True


# === Load Tier-1 v1 corpus ===

### === PDF loader ===

In [9]:
def load_tier1():
    docs = []
    for pdf in sorted(TIER1_DIR.glob("*.pdf")):
        d = fitz.open(pdf); text = "\n".join(pg.get_text() for pg in d); d.close()
        docs.append({"source": pdf.stem, "tier": "tier1", "tier1_v": "v1", "text": text})
        print(f"  {pdf.stem[:58]:58s} {len(text):>9,} chars")
    return docs
tier1_docs = load_tier1()

  2017 ACCAHAHRS — Evaluation of Patients with Syncope         361,430 chars

  2017 AHA_ACC_HRS — Management of Ventricular Arrhythmias     687,839 chars
  ESC 2018 — Guidelines for the Diagnosis and Management of    359,141 chars


  ESC 2022 — Guidelines for Ventricular Arrhythmias & SCD      725,258 chars


# === Load Tier-1 v2 (new wearable-relevant guidelines) ===

### === Text loader + provenance manifest ===

In [10]:
tier1_v2_docs = []
for t in sorted(TIER1_V2_DIR.glob("*.txt")):
    tier1_v2_docs.append({"source": t.stem, "tier": "tier1", "tier1_v": "v2",
                          "text": t.read_text(encoding="utf-8")})
    print(f"  {t.stem[:58]:58s} {len(tier1_v2_docs[-1]['text']):>9,} chars")
pd.read_csv(TIER1_V2_DIR/"manifest.csv")[["slug","status","provenance","license"]]

  accaha2023_af_guideline                                      771,946 chars
  ehra2022_digital_devices_arrhythmias                         140,248 chars


,slug,status,provenance,license
0,ehra2022_digital_devices_arrhythmias,included,PMC HTML https://pmc.ncbi.nlm.nih.gov/articles...,free-to-read (non-OA deposit)
1,accaha2023_af_guideline,included,NCBI-efetch PMC11104284,open-access full text (API)
2,esc2024_af_guideline,excluded,landing page: HTTPError,unknown


# === Load Tier-2 corpus ===

### === Markdown loader ===

In [11]:
def load_tier2():
    docs = []
    for bdir in sorted(TIER2_DIR.iterdir()):
        if not bdir.is_dir(): continue
        mds = sorted(bdir.glob("*.md"))
        for m in mds:
            docs.append({"source": m.stem, "tier": "tier2", "bucket": bdir.name,
                         "text": m.read_text(encoding="utf-8")})
        print(f"  {bdir.name:28s} {len(mds):>3} articles")
    return docs
tier2_docs = load_tier2()

  01_ppg_arrhythmia             50 articles
  02_ppg_signal_quality         25 articles
  03_ecg_anomaly_ml             45 articles
  04_wearable_stress            35 articles
  05_continuous_monitoring      25 articles
  06_biosignal_methods          20 articles


# === Chunk corpus ===

### === Chunking function (v1-identical) ===

In [12]:
def chunk_text(text, chunk_words=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    words = text.split()
    if len(words) <= chunk_words: return [text]
    chunks, start = [], 0
    while start < len(words):
        chunks.append(" ".join(words[start:start+chunk_words])); start += chunk_words - overlap
    return chunks

### === Chunk all documents ===

In [13]:
all_chunks = []
for doc in tier1_docs + tier1_v2_docs + tier2_docs:
    for i, ch in enumerate(chunk_text(doc["text"])):
        all_chunks.append({"text": ch, "source": doc["source"], "tier": doc["tier"],
                           "tier1_v": doc.get("tier1_v",""), "bucket": doc.get("bucket",""), "chunk_idx": i})
print(f"{len(all_chunks):,} chunks (mean {np.mean([len(c['text'].split()) for c in all_chunks]):.0f} words)")
print("  guideline chunks:", sum(1 for c in all_chunks if c["tier"]=="tier1"))

5,045 chunks (mean 489 words)
  guideline chunks: 992


# === Embed + Index (ChromaDB) ===

In [14]:
embedder = SentenceTransformer(EMBED_MODEL)
col = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_or_create_collection("medical_corpus_v2")
if col.count() == 0:
    t0 = time.time(); embs = []
    for i in range(0, len(all_chunks), 64):
        embs.extend(embedder.encode([c["text"] for c in all_chunks[i:i+64]]).tolist())
    col.add(embeddings=embs, documents=[c["text"] for c in all_chunks],
           metadatas=[{k:v for k,v in c.items() if k!="text"} for c in all_chunks],
           ids=[f"chunk_{i}" for i in range(len(all_chunks))])
    print(f"embedded {len(all_chunks):,} chunks in {time.time()-t0:.0f}s")
else:
    print(f"collection already populated ({col.count():,} chunks) — skipping embed (v1 convention)")

collection already populated (5,045 chunks) — skipping embed (v1 convention)


# === Sanity retrieval test ===

### === retrieve (dense + source-diversity) ===

In [15]:
def retrieve(query, top_k=TOP_K, pool=POOL, max_per_source=1):
    res = col.query(query_embeddings=embedder.encode([query]).tolist(), n_results=pool,
                    include=["documents","metadatas","distances"])
    metas = res["metadatas"][0]
    picked, counts = [], {}
    for i, m in enumerate(metas):
        cnt = counts.get(m["source"], 0)
        if cnt >= max_per_source: continue
        picked.append(i); counts[m["source"]] = cnt+1
        if len(picked) == top_k: break
    if len(picked) < top_k:
        for i in range(len(metas)):
            if i not in picked: picked.append(i)
            if len(picked) == top_k: break
    return {"sources": [metas[i]["source"] for i in picked],
            "tiers":   [metas[i]["tier"] for i in picked],
            "docs":    [res["documents"][0][i] for i in picked],
            "dists":   [res["distances"][0][i] for i in picked]}

### === 3 test queries ===

In [16]:
for q in ["How does PPG detect atrial fibrillation?",
              "What causes false alarms in wearable heart rate monitors?",
              "How is stress detected from electrodermal activity?"]:
    r = retrieve(q, top_k=3)
    print("QUERY:", q)
    for s, t, d in zip(r["sources"], r["tiers"], r["dists"]):
        print(f"   [{t}] {s[:70]} (d={d:.3f})")
    print()

QUERY: How does PPG detect atrial fibrillation?
   [tier1] ehra2022_digital_devices_arrhythmias (d=0.677)
   [tier2] PMC12635274_fibricheck_detection_capabilities_for_atrial_fibrillation_ (d=0.702)
   [tier2] PMC12925684_diagnostic_performance_of_two_commercially_available_ppgba (d=0.712)

QUERY: What causes false alarms in wearable heart rate monitors?
   [tier2] PMC13027176_the_use_of_digital_devices_in_the_management_of_athletes_w (d=0.726)
   [tier2] PMC12986385_atrial_fibrillation_and_cognitive_decline_a_systematic_rev (d=0.749)
   [tier1] ehra2022_digital_devices_arrhythmias (d=0.750)

QUERY: How is stress detected from electrodermal activity?
   [tier2] PMC13211236_electrodermal_temperatureadjusted_electrodermal_activity_e (d=0.653)
   [tier2] PMC13076599_supervised_information_gainbased_feature_selection_for_mul (d=0.675)
   [tier2] PMC12608435_shortterm_detection_of_dynamic_stress_levels_in_exergaming (d=0.679)



# === Load flagged windows + build v2 queries ===

### === Load the handoff (v1 archive) ===

In [17]:
flagged = pd.read_parquet(PROJECT_ROOT/"outputs_v1_archive/flagged_windows.parquet")
print(f"{len(flagged)} flagged windows | IF {int(flagged.flag_if.sum())}, "
      f"LOF {int(flagged.flag_lof.sum())}, both {int(flagged.flag_both.sum())}")

398 flagged windows | IF 216, LOF 216, both 34


### === Load the full PPG-DaLiA feature matrix ===

In [18]:
z = np.load(OUTPUT_DIR/"cache/ppgdalia.npz", allow_pickle=True)
X_all, subj_all = z["X"], z["subject"]
cols = [f"{ch}__{fn}" for ch in CHANNELS
        for fn in ["mean","std","min","max","ptp","median","skew","kurt","p25","p75","up_ratio","roughness"]]
df_all = pd.DataFrame(X_all, columns=cols); df_all["subject"] = subj_all
df_all["window_idx"] = np.arange(len(df_all))
print(f"{len(df_all)} windows x {len(cols)} features, {df_all.subject.nunique()} subjects")

4308 windows x 60 features, 15 subjects


### === Map archived flags exactly onto the matrix ===

In [19]:
df_all["flag_if"] = False; df_all["flag_lof"] = False
offsets, off = {}, 0
for s in sorted(df_all.subject.unique()):
    offsets[s] = off; off += int((df_all.subject==s).sum())
for _, r in flagged.iterrows():
    g = offsets[int(r["subject"])] + int(r["window_idx"])
    df_all.loc[g, "flag_if"] = bool(r["flag_if"]); df_all.loc[g, "flag_lof"] = bool(r["flag_lof"])
print(f"IF {int(df_all.flag_if.sum())}, LOF {int(df_all.flag_lof.sum())}, "
      f"both {int((df_all.flag_if & df_all.flag_lof).sum())}, "
      f"union {int((df_all.flag_if | df_all.flag_lof).sum())}  (archive: 216/216/34/398)")

IF 216, LOF 216, both 34, union 398  (archive: 216/216/34/398)


### === Query builder v2 (correct reference class) ===

In [20]:
TOPIC_PHRASES = {
    "ecg": "electrocardiogram rhythm irregularity heart rate variability arrhythmia ectopic beats atrial fibrillation",
    "resp": "respiration rate breathing pattern tachypnea bradypnea ventilation",
    "bvp": "photoplethysmography pulse waveform amplitude perfusion signal quality motion artifact atrial fibrillation screening",
    "wrist_eda": "electrodermal activity skin conductance sympathetic stress arousal sweat response",
    "wrist_temp": "skin temperature thermal perfusion vasomotor ambient temperature sensor effects"}

def subject_reference(df, subject):
    """This subject's NON-FLAGGED windows — the normal population the detector
    learned, implementable online via running stats (fixes v1's batch, cross-subject
    flagged-window reference)."""
    sub = df[df.subject == subject]
    normal = sub[(~sub.flag_if) & (~sub.flag_lof)]
    return {f"{ch}__{feat}": (normal[f"{ch}__{feat}"].mean(), normal[f"{ch}__{feat}"].std(ddof=0) or 1e-9)
            for ch in CHANNELS for feat in ["mean","kurt","ptp"]}

def build_query_v2(row, ref):
    zs  = {ch: (row[f"{ch}__mean"]-ref[f"{ch}__mean"][0]) / ref[f"{ch}__mean"][1] for ch in CHANNELS}
    kzs = {ch: (row[f"{ch}__kurt"]-ref[f"{ch}__kurt"][0]) / ref[f"{ch}__kurt"][1] for ch in CHANNELS}
    pzs = {ch: (row[f"{ch}__ptp"] -ref[f"{ch}__ptp"][0])  / ref[f"{ch}__ptp"][1]  for ch in CHANNELS}
    top2 = sorted(CHANNELS, key=lambda ch: -abs(zs[ch]))[:2]
    if row.flag_if and row.flag_lof: parts = ["Biosignal window flagged by both anomaly detectors (Isolation Forest and LOF)."]
    elif row.flag_if:                parts = ["Biosignal window flagged by the Isolation Forest anomaly detector."]
    else:                            parts = ["Biosignal window flagged by the LOF anomaly detector."]
    shape = max(max(kzs[ch], pzs[ch]) for ch in top2); shift = max(abs(zs[ch]) for ch in top2)
    if shape > 1.5:   parts.append("Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline).")
    elif shift > 1.5: parts.append("Deviating channels show a sustained level shift from this subject's baseline.")
    elif shift > 1.0: parts.append("Deviating channels show a moderate shift from this subject's baseline.")
    else:             parts.append("Deviating channels are only mildly unusual vs this subject's baseline.")
    for ch in top2:
        parts.append(f"{'elevated' if zs[ch]>0 else 'reduced'} {ch} (z={zs[ch]:+.1f} vs subject baseline, mean={row[f'{ch}__mean']:.2f})")
    parts.append("Relevant topics: " + " ".join(TOPIC_PHRASES[ch] for ch in top2) + ".")
    parts.append("Other readings: " + ", ".join(f"{ch} mean={row[f'{ch}__mean']:.2f}" for ch in CHANNELS if ch not in top2) + ".")
    return " ".join(parts)

### === Show the first 3 v2 queries ===

In [21]:
union = df_all[df_all.flag_if | df_all.flag_lof]
for _, row in union.head(3).iterrows():
    q = build_query_v2(row, subject_reference(df_all, row.subject))
    print(f"S{int(row.subject)} w{int(row.window_idx)}: {q[:210]}...\n")

S1 w1: Biosignal window flagged by the LOF anomaly detector. Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline). elevated bvp (z=+0.8 vs subject basel...

S1 w6: Biosignal window flagged by the LOF anomaly detector. Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline). elevated resp (z=+1.0 vs subject base...

S1 w13: Biosignal window flagged by the LOF anomaly detector. Deviating channels show an abrupt, high-amplitude pattern (elevated kurtosis/peak-to-peak vs this subject's baseline). reduced wrist_eda (z=-0.6 vs subject ...



### === Sanity check: the corrected z-metric separates stress on WESAD ===

In [22]:
from scipy.stats import mannwhitneyu
w = np.load(OUTPUT_DIR/"cache/wesad.npz", allow_pickle=True)
stress_nn, baseline_nn = [], []
for s in w["subjects"]:
    s = int(s); X, y = w[f"X_{s}"], w[f"y_{s}"]
    dfw = pd.DataFrame(X, columns=cols); base = dfw[y==1]
    ref = {ch: (base[f"{ch}__mean"].mean(), base[f"{ch}__mean"].std(ddof=0) or 1e-9) for ch in CHANNELS}
    for mask, acc in [(y==2, stress_nn), (y==1, baseline_nn)]:
        for _, r in dfw[mask].iterrows():
            zz = [abs((r[f"{ch}__mean"]-ref[ch][0])/ref[ch][1]) for ch in CHANNELS]
            acc.append(sorted(zz)[-2:])
st, bt = np.array(stress_nn).max(axis=1), np.array(baseline_nn).max(axis=1)
print(f"stress max|z| mean {st.mean():.1f} vs baseline {bt.mean():.1f} — p = {mannwhitneyu(st, bt, alternative='greater').pvalue:.3g}")

stress max|z| mean 43.7 vs baseline 1.5 — p = 2.54e-138


### === Build all 398 queries + retrieval ===

In [23]:
p = OUTPUT_DIR/"alerts_retrieval_v2.jsonl"
if p.exists():
    alerts = [json.loads(l) for l in open(p, encoding="utf-8")]
    print(f"loaded {len(alerts)} cached retrievals")
else:
    alerts = []
    for _, row in union.iterrows():
        q = build_query_v2(row, subject_reference(df_all, row.subject))
        r = retrieve(q)
        alerts.append({"subject": int(row.subject), "window_idx": int(row.window_idx), "query": q,
                       "flag_if": bool(row.flag_if), "flag_lof": bool(row.flag_lof),
                       "sources": r["sources"], "tiers": r["tiers"],
                       "context": "\n\n---\n\n".join(f"[{s}]\n{d}" for s, d in zip(r["sources"], r["docs"]))})
    with open(p, "w", encoding="utf-8") as f:
        for a in alerts: f.write(json.dumps(a)+"\n")
    print(f"built + saved {len(alerts)} retrievals")
t1 = sum(1 for a in alerts if any(t=="tier1" for t in a["tiers"]))
print(f"unique docs used: {len({s for a in alerts for s in a['sources']})} | "
      f"guideline reach: {t1}/{len(alerts)} ({100*t1/len(alerts):.1f}%)")

loaded 398 cached retrievals
unique docs used: 44 | guideline reach: 70/398 (17.6%)


# === Labeled events (labels NEVER enter the queries) ===

### === WESAD: top-50 LOF-scored stress windows ===

In [24]:
DS1 = ["101","106","108","109","112","114","115","116","118","119","122","124",
       "201","203","205","207","208","209","215","220","223","230"]
DS2 = ["100","103","105","111","113","117","121","123","200","202","210","212",
       "213","214","219","221","222","228","231","232","233","234"]
AAMI2 = {"N","L","R","e","j"}; BEAT_SYMBOLS2 = AAMI2 | {"A","a","J","S","V","E","F","f","Q","/","!"}
MITBIH_DIR = PROJECT_ROOT/"Dataset/mit-bih-arrhythmia-database-1.0.0/mit-bih-arrhythmia-database-1.0.0"

Xb = np.vstack([w[f"X_{s}"][w[f"y_{s}"]==1] for s in w["subjects"]])
sc_w = StandardScaler().fit(Xb)
lof_w = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.15).fit(sc_w.transform(Xb))
cand = []
for s in w["subjects"]:
    s = int(s); Xs = w[f"X_{s}"][w[f"y_{s}"]==2]
    for i, scr in enumerate(-lof_w.score_samples(sc_w.transform(Xs))): cand.append((float(scr), s, i))
cand.sort(reverse=True)
print(f"WESAD: picked 50 stress windows (top LOF scores {cand[0][0]:.2f}..{cand[49][0]:.2f})")

WESAD: picked 50 stress windows (top LOF scores 5.41..2.08)


### === MIT-BIH: annotation-driven window selection ===

In [25]:
# NOTE: a first-pass selection ranked windows by DETECTOR flags — but the inter-patient
# detector is near chance, so 49/50 picked windows had NO annotated ectopy. Superseded
# (see superseded_keys.json); v2 selects by expert annotations (query stays detector-side).
z_m = np.load(OUTPUT_DIR/"cache/mitbih.npz", allow_pickle=True)
X_m2, y_m2, rec_m2 = z_m["X"], z_m["y"], z_m["record"].astype(str)
trm = np.isin(rec_m2, DS1) & (y_m2==0); tem = np.isin(rec_m2, DS2)
sc_m = StandardScaler().fit(X_m2[trm])
lof_m = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.15).fit(sc_m.transform(X_m2[trm]))
se_te = -lof_m.score_samples(sc_m.transform(X_m2[tem]))
thr_m = np.percentile(-lof_m.score_samples(sc_m.transform(X_m2[trm])), 85)
flag_abs = np.zeros(len(X_m2), bool); flag_abs[np.where(tem)[0][se_te > thr_m]] = True

cands = {"VEB": [], "SVEB": []}
for rec in DS2:
    sig, _ = wfdb.rdsamp(str(MITBIH_DIR/rec))
    ann = wfdb.rdann(str(MITBIH_DIR/rec), "atr")
    idx_te = np.where(tem & (rec_m2==rec))[0]
    seq = []
    for i, sym in zip(ann.sample, ann.symbol):
        if sym not in BEAT_SYMBOLS2: continue
        st_, en_ = i-144, i+144
        if st_ < 0 or en_ > sig.shape[0]: continue
        seq.append((i, sym, len(seq)))
    row_of = {pos: int(r) for pos, r in zip([x[2] for x in seq], idx_te)}
    wins = {}
    for sample, sym, pos in seq:
        wd = wins.setdefault(sample//(360*30), {"rows": [], "syms": []})
        wd["rows"].append(row_of[pos]); wd["syms"].append(sym)
    for widx, wd in wins.items():
        abn = [s_ for s_ in wd["syms"] if s_ not in AAMI2]
        if len(abn) < 3 or len(wd["rows"]) < 5: continue
        cls = "VEB" if any(s_ in {"V","E"} for s_ in abn) else "SVEB"
        cands[cls].append((len(abn), rec, widx, wd, abn))
print(f"MIT-BIH candidates (>=3 annotated abnormal beats): VEB {len(cands['VEB'])}, SVEB {len(cands['SVEB'])} → picked 25+25")

MIT-BIH candidates (>=3 annotated abnormal beats): VEB 357, SVEB 101 → picked 25+25


### === PTB-XL: stratified top-scored pathology per superclass ===

In [26]:
zp = np.load(OUTPUT_DIR/"cache/ptbxl.npz", allow_pickle=True)
Xp2, yp2, fp2 = zp["X"], zp["y"], zp["fold"]
trp = (fp2<=8) & (yp2==0)
sc_p = StandardScaler().fit(Xp2[trp])
lof_p = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.15).fit(sc_p.transform(Xp2[trp]))
se_fold10 = -lof_p.score_samples(sc_p.transform(Xp2[fp2==10]))

db = pd.read_csv(PROJECT_ROOT/"Dataset/ptb-xl-1.0.3/ptbxl_database.csv")
scp = pd.read_csv(PROJECT_ROOT/"Dataset/ptb-xl-1.0.3/scp_statements.csv", index_col=0)
db["super"] = db.scp_codes.apply(lambda cp: {scp.loc[c,"diagnostic_class"] for c in ast.literal_eval(cp)
                                             if c in scp.index and isinstance(scp.loc[c,"diagnostic_class"], str)})
db["y"] = db.super.apply(lambda st: 0 if st=={"NORM"} else (-1 if not st else 1))
dbf = db[db.y != -1]                       # same filter as the cache — keeps alignment
f10 = dbf[dbf.strat_fold==10].reset_index()
assert len(f10) == len(se_fold10)
score_by_ecg = {int(f10.iloc[k].ecg_id): float(se_fold10[k]) for k in range(len(f10))}
picked_ptbxl = []
for cls in ["MI","STTC","CD","HYP"]:
    cdb = f10[f10.super.apply(lambda st: cls in st)].copy()
    cdb["score"] = cdb.ecg_id.map(lambda e: score_by_ecg.get(e, -9.0))
    for r in cdb.sort_values("score", ascending=False).head(12).itertuples():
        picked_ptbxl.append((r.ecg_id, r.filename_lr, cls))
print(f"PTB-XL: {len(picked_ptbxl)} records:", dict(Counter(c for _,_,c in picked_ptbxl)))

PTB-XL: 48 records: {'MI': 12, 'STTC': 12, 'CD': 12, 'HYP': 12}


# === Grounded explanation generator ===

### === System prompt (v1-identical rules) ===

In [27]:
SYSTEM_PROMPT_150 = """You are a clinical decision-support assistant that explains wearable biosignal anomalies.
You receive:
1. A description of an anomaly detected in a 30-second window of wearable signals.
2. Retrieved excerpts from peer-reviewed clinical guidelines and research articles.
STRICT RULES (never violate):
- Answer ONLY using the provided retrieved context.
- Cite the source document for every clinical claim. Format: [Source Name].
- If the retrieved context does not cover the anomaly, say: "The retrieved context is insufficient to explain this pattern."
- NEVER invent facts, numbers, citations, or medical conclusions not present in the context.
- This is a research tool, NOT a diagnostic device. State this once at the end.
- Keep the explanation under 150 words. Use plain language a nurse could understand.
Output format:
DETECTED: [one-sentence summary of what the anomaly pattern suggests]
EVIDENCE: [what the guidelines/literature say, with citations]
RECOMMENDATION: [what clinical follow-up the guidelines suggest, or "context insufficient"]
DISCLAIMER: Research decision-support tool. Not a diagnostic device. Does not replace clinical judgment."""

### === Canonicalizer (capture rule FIXED vs v1) ===

In [28]:
def canonicalize_fixed(raw_text, sources):
    """v1-intended canonicalizer with the ID capture fixed: the citation ID is the
    PMC token; trailing slug text is display noise (v1's greedy regex dropped valid
    citations written as [PMC123_full_title]). Every snap is logged."""
    valid = sorted({s.split("_")[0] for s in sources if s.startswith("PMC")})
    snaps = []
    def _fix(m):
        cid = m.group(1)
        if cid in valid: return f"[{cid}]"
        close = difflib.get_close_matches(cid, valid, n=1, cutoff=0.75)
        if close:
            snaps.append({"before": cid, "after": close[0]}); return f"[{close[0]}]"
        snaps.append({"before": cid, "after": None}); return ""
    return re.sub(r"\[(PMC\d+)[^\]]*\]", _fix, raw_text), snaps

### === generate_one ===

In [29]:
def generate_one(query, context, model=LLM_MODEL, prompt=SYSTEM_PROMPT_150, num_predict=500):
    resp = ollama.chat(model=model, think=False,
        messages=[{"role":"system","content":prompt},
                  {"role":"user","content":f"ANOMALY:\n{query}\n\nRETRIEVED CONTEXT:\n{context}"}],
        options={"temperature":0.1,"num_predict":num_predict,"num_ctx":10000,"num_gpu":99})
    return resp["message"]["content"].strip()

### === Test on 3 flagged windows ===

In [30]:
for a in alerts[:3]:
    raw = generate_one(a["query"], a["context"])
    fixed, snaps = canonicalize_fixed(raw, a["sources"])
    print(f"--- S{a['subject']} w{a['window_idx']} ({len(snaps)} citation repairs/drops) ---")
    print(fixed[:520], "\n")

--- S1 w1 (0 citation repairs/drops) ---
DETECTED: The abrupt, high-amplitude pattern in PPG signals suggests potential motion artifact rather than a true cardiac event like atrial fibrillation or tachypnea.

EVIDENCE: Guidelines indicate that abnormal readings from consumer heart rate devices (especially those using PPG technology) should be critically evaluated to distinguish suspected arrhythmia noise or oversensing caused by artifacts [ehra2022_digital_devices_arrhythmias]. Furthermore, wearable ECG algorithms can produce inconclusive recordings due t 



--- S1 w6 (0 citation repairs/drops) ---
DETECTED: The abrupt high-amplitude signal suggests a sudden respiratory event or arousal, potentially indicating tachypnea (rapid breathing) or an obstructive sleep apnea episode.
EVIDENCE: Guidelines note that changes in pulse transit time and heart rate serve as markers for arousals from sleep [PMC13212801]. Additionally, wrist-worn devices can detect oxygen desaturation events relevant to diagnosing conditions like obstructive sleep apnea when combined with standardized signal processing [PMC12912879].
RECOMMEN 



--- S1 w13 (0 citation repairs/drops) ---
DETECTED: The pattern suggests a physiological response where reduced skin temperature indicates vasoconstriction (blood flow restriction) while low EDA implies minimal sweat activity, potentially indicating sensor drift or non-stressful thermal changes rather than acute sympathetic arousal [PMC12635167].

EVIDENCE: Skin temperature varies based on skin blood flow and ambient conditions; reduced readings can occur due to environmental factors like high humidity which elevate baseline conductance independently of st 



# === Batch generation (resume-safe) ===

### === Load the completed run (the cold-start loop is in scripts/v2/rag_pipeline_v2.py) ===

In [31]:
rows = [json.loads(l) for l in open(GEN_JSONL, encoding="utf-8")]
sup = json.loads((OUTPUT_DIR/"superseded_keys.json").read_text())["superseded_mitbih_keys"]
rows = [r for r in rows if r["key"] not in sup]
print(f"{len(rows)} explanations (50 superseded detector-ranked MIT-BIH keys excluded)")
pd.DataFrame([{k: r.get(k) for k in ("group","subgroup","model","prompt")} for r in rows]
            ).value_counts(["group","subgroup","model","prompt"]).to_frame("n")

646 explanations (50 superseded detector-ranked MIT-BIH keys excluded)


n
group  subgroup    model       prompt     
dalia  main        qwen3.5:9b  150     398
       genablation llama3.1:8b 150      50
       wordcap     qwen3.5:9b  300      50
mitbih labeled     qwen3.5:9b  150      50
wesad  labeled     qwen3.5:9b  150      50
ptbxl  labeled     qwen3.5:9b  150      48

### === Generation latency ===

In [32]:
mn = [r for r in rows if r["group"]=="dalia" and r["subgroup"]=="main"]
lat = np.array([r["latency_sec"] for r in mn])
print(f"{len(lat)} alerts | mean {lat.mean():.1f}s | median {np.median(lat):.1f}s | max {lat.max():.1f}s (RTX 5060 laptop)")

398 alerts | mean 11.0s | median 10.8s | max 46.0s (RTX 5060 laptop)


# === Citation accuracy (programmatic, raw AND repaired) ===

### === Audit loop ===

In [33]:
cit_re = re.compile(r"\[(PMC\d+)[^\]]*\]")
name_re = re.compile(r"\[([^]\[]+)\]")
stats = {"raw_cit":0, "raw_ok":0, "rep_cit":0, "rep_ok":0, "snaps":0, "drops":0, "t1_ok":0, "t1_bad":0}
for r in [x for x in rows if x["subgroup"]=="main"]:
    valid = {s.split("_")[0] for s in r["sources"] if s.startswith("PMC")}
    t1 = [s for s in r["sources"] if not s.startswith("PMC")]
    fixed, snaps = canonicalize_fixed(r["raw_explanation"], r["sources"])
    stats["snaps"] += sum(1 for s in snaps if s["after"])
    stats["drops"] += sum(1 for s in snaps if not s["after"])
    for which, txt in [("raw", r["raw_explanation"]), ("rep", fixed)]:
        cits = [m.group(1) for m in cit_re.finditer(txt)]
        stats[f"{which[:3]}_cit"] += len(cits)
        stats[f"{which[:3]}_ok"]  += sum(1 for c in cits if c in valid)
    for m in name_re.finditer(fixed):
        tok = m.group(1).strip()
        if tok.startswith("PMC"): continue
        if any(s.startswith(tok) or tok.startswith(s) for s in t1): stats["t1_ok"] += 1
        else: stats["t1_bad"] += 1
print(f"RAW:      {stats['raw_cit']} citations, {stats['raw_ok']} valid = {100*stats['raw_ok']/stats['raw_cit']:.2f}%")
print(f"REPAIRED: {stats['rep_cit']} citations, {stats['rep_ok']} valid = {100*stats['rep_ok']/stats['rep_cit']:.2f}%")
print(f"canonicalizer: {stats['snaps']} snapped, {stats['drops']} dropped (genuine fabrications)")
print(f"guideline name-citations: {stats['t1_ok']} valid, {stats['t1_bad']} unmatched")

RAW:      1208 citations, 1196 valid = 99.01%
REPAIRED: 1196 citations, 1196 valid = 100.00%
canonicalizer: 0 snapped, 12 dropped (genuine fabrications)
guideline name-citations: 109 valid, 8 unmatched


# === Judge validation (corruption benchmark) ===

### === Build the corruption set ===

In [34]:
import random as _rnd; _rnd.seed(7)
FABRICATED_FACTS = [
    "This pattern is associated with a 3-fold increase in 30-day mortality according to the sources.",
    "The literature states that patients with this finding require immediate hospital admission.",
    "According to the retrieved context, this signal pattern predicts stroke with 92% probability.",
    "The sources confirm that this finding always progresses to cardiac arrest if untreated.",
    "Retrieved guidelines recommend starting anticoagulation immediately for this pattern."]
main_rows = [r for r in rows if r["subgroup"]=="main"
             and len(re.findall(r"\[PMC\d+", r["explanation"])) >= 2]
sample = _rnd.sample(main_rows, 100)
bench = []
for i, r in enumerate(sample):
    txt = r["explanation"]; cits = re.findall(r"PMC\d+", txt)
    ctype = i % 4; corrupted = txt
    if ctype == 0 and len(cits) >= 1:
        others = [s.split("_")[0] for s in r["sources"] if s.split("_")[0] not in cits[:1]]
        if others: corrupted = txt.replace(f"[{cits[0]}]", f"[{others[0]}]", 1)
        else: ctype = 1
    if ctype == 1:   corrupted = txt.replace("EVIDENCE:", f"EVIDENCE: {_rnd.choice(FABRICATED_FACTS)} ", 1)
    elif ctype == 2: corrupted = txt.replace(f"[{cits[0]}]", "[PMC99999999]", 1)
    elif ctype == 3: corrupted = txt.replace("DETECTED:", "DETECTED: This finding is diagnostic of acute myocardial infarction and requires emergency treatment. ", 1)
    bench.append({"row": r, "clean": txt, "corrupted": corrupted,
                  "ctype": ["citation_swap","fabricated_fact","fabricated_citation","diagnostic_exaggeration"][ctype]})
print("100 corrupted + 100 clean controls | example (", bench[0]["ctype"], "):")
print(bench[0]["corrupted"][:260], "...")

100 corrupted + 100 clean controls | example ( citation_swap ):
DETECTED: The pattern suggests a physiological response to thermal stress or vasomotor change rather than respiratory failure, given elevated skin temperature alongside reduced respiration signal amplitude.

EVIDENCE: Wearable biosignals can be modified by loc ...


### === Judge prompt + local judge ===

In [35]:
JUDGE_PROMPT = open(PROJECT_ROOT/"scripts/v2/judge_prompt_v2.txt", encoding="utf-8").read()

def local_judge(model, query, explanation, context):
    # think=False is essential for gemma4: default thinking mode consumes the
    # token budget and returns EMPTY content (see *_INVALID_empty_thinking.csv)
    resp = ollama.chat(model=model, think=False,
        messages=[{"role":"system","content":JUDGE_PROMPT},
                  {"role":"user","content":f"QUERY: {query}\nSOURCES:\n{context}\nEXPLANATION:\n{explanation}"}],
        options={"temperature":0.1,"num_predict":300,"num_ctx":6000,"num_gpu":99})
    scores = {}
    for line in str(resp["message"]["content"]).split("\n"):
        m = re.match(r"(FAITHFULNESS|RELEVANCE|COMPLETENESS):\s*([123])", line.strip(), re.I)
        if m: scores[m.group(1).lower()] = int(m.group(2))
    return scores

### === Spot-check: gemma4 on 2 corrupted + 2 clean ===

In [36]:
_alert_ctx = {f"S{a['subject']}|w{a['window_idx']}": a["context"] for a in alerts}
def ctx_of(row):
    if row["key"].startswith("dalia|"):
        return _alert_ctx[row["key"].split("|",1)[1]]
    return retrieve(row["query"])["context"]

for b in bench[:2]:
    print("corrupted:", b["ctype"], "->", local_judge("gemma4:e4b", b["row"]["query"], b["corrupted"], ctx_of(b["row"])))
for b in bench[:2]:
    print("clean    :", local_judge("gemma4:e4b", b["row"]["query"], b["clean"], ctx_of(b["row"])))

corrupted: citation_swap -> {}


corrupted: fabricated_fact -> {'faithfulness': 1, 'relevance': 3, 'completeness': 2}


clean    : {}


clean    : {'faithfulness': 2, 'relevance': 3, 'completeness': 2}


### === Full benchmark results (200 calls per judge, cached CSVs) ===

In [37]:
for model in JUDGE_CANDIDATES:
    d = pd.read_csv(OUTPUT_DIR/f"judge_validation_{model.replace(':','_').replace('/','_')}.csv")
    det = (d[(d.is_corrupted==1) & (d.faithfulness==1)].shape[0]) / (d.is_corrupted==1).sum()
    fp  = (d[(d.is_corrupted==0) & (d.faithfulness==1)].shape[0]) / (d.is_corrupted==0).sum()
    by_type = d[d.is_corrupted==1].groupby("ctype").apply(lambda g: (g.faithfulness==1).mean(), include_groups=False).round(2).to_dict()
    print(f"{model}: detection {det:.2f}, FP {fp:.2f}, by type {by_type}")

llama3.1:8b: detection 0.00, FP 0.00, by type {'citation_swap': 0.0, 'diagnostic_exaggeration': 0.0, 'fabricated_citation': 0.0, 'fabricated_fact': 0.0}
gemma4:e4b: detection 0.48, FP 0.01, by type {'citation_swap': 0.0, 'diagnostic_exaggeration': 0.96, 'fabricated_citation': 0.16, 'fabricated_fact': 0.8}


# === Main judging (validated local judge) ===

### === Scores by group ===

In [38]:
ev = pd.read_csv(OUTPUT_DIR/"rag_evaluation_v2.csv")
ev.groupby("subgroup").agg(n=("local_faithfulness","size"),
                           faith=("local_faithfulness","mean"),
                           relev=("local_relevance","mean"),
                           compl=("local_completeness","mean")).round(2)

,n,faith,relev,compl
subgroup,,,,
genablation,50,1.78,2.34,1.52
labeled,148,2.53,3.00,2.54
main,398,2.21,2.65,2.20
wordcap,50,2.02,2.52,2.12


### === Faithfulness distribution on the 398 wearable alerts ===

In [39]:
m = ev[(ev.group=="dalia") & (ev.subgroup=="main")]
print("score -> count:", m.local_faithfulness.value_counts().sort_index().to_dict(), "(0 = parse failure)")

score -> count: {0: 45, 1: 2, 2: 176, 3: 175} (0 = parse failure)


# === Labeled-event concordance ===

### === Lexicons + concordance loop ===

In [40]:
LEXICONS = {
 "stress":["stress","arousal","sympathetic","anxiety","mental load","psychological","emotional"],
 "VEB":["ventricular","pvc","premature ventricular","ventricular tachycard"],
 "SVEB":["supraventricular","atrial premature","pac","atrial ectopy","premature atrial","atrial fibrillation","atrial tachyarrhythm"],
 "MI":["infarct","ischemi","stemi","coronary occlusion","st-elevation","st elevation"],
 "STTC":["repolarization","st depression","st-segment","st segment","t-wave","t wave inversion"],
 "CD":["conduction","bundle branch","heart block","av block","pr interval"],
 "HYP":["hypertroph","chamber enlargement","left ventricular mass"]}
ARTIFACT_TERMS = ["artifact","motion","sensor displacement","sensor contact","signal quality",
                  "electrode","noise","poor contact","device"]

def sec(t, a, b):
    mm = re.search(rf"{a}:\s*(.*?)(?={b}:|$)", t, re.S)
    return mm.group(1).strip().lower() if mm else ""

table = {}
for grp in ("wesad","mitbih","ptbxl"):
    sel = [r for r in rows if r["subgroup"]=="labeled" and r["group"]==grp]
    st = {"n":len(sel),"concordant":0,"artifact":0,"insufficient":0,"other":0,"by_label":{}}
    for r in sel:
        det = sec(r["explanation"], "DETECTED", "EVIDENCE"); lab = r["true_label"]
        st["by_label"].setdefault(lab, {"n":0,"ok":0}); st["by_label"][lab]["n"] += 1
        if any(t in det for t in LEXICONS.get(lab, [])):
            st["concordant"] += 1; st["by_label"][lab]["ok"] += 1
            if any(t in det for t in ARTIFACT_TERMS): st["artifact"] += 1
        elif "insufficient" in det: st["insufficient"] += 1
        elif any(t in det for t in ARTIFACT_TERMS): st["artifact"] += 1
        else: st["other"] += 1
    table[grp] = st

pd.DataFrame({g: {"n":v["n"], "concordant":f'{v["concordant"]} ({100*v["concordant"]/v["n"]:.0f}%)',
                  "artifact language":v["artifact"], "other":v["other"]} for g,v in table.items()}).T

,n,concordant,artifact language,other
wesad,50,47 (94%),28,1
mitbih,50,6 (12%),28,22
ptbxl,48,3 (6%),20,25


# === Near-duplicates: before/after the v2 fixes ===

### === Embed explanations + cluster at cosine 0.9 ===

In [41]:
texts = [sec(r["explanation"],"DETECTED","EVIDENCE") + "\n" +
         sec(r["explanation"],"EVIDENCE","RECOMMENDATION")
         for r in rows if r["group"]=="dalia" and r["subgroup"]=="main"]
emb = embedder.encode(texts, normalize_embeddings=True)
sim = emb @ emb.T; np.fill_diagonal(sim, -1)
nn = sim.max(axis=1)
unassigned, cluster, cid = set(range(len(texts))), np.full(len(texts), -1), 0
while unassigned:
    seed = min(unassigned)
    members = [j for j in unassigned if sim[seed, j] > 0.9] + [seed]
    for j in members: cluster[j] = cid; unassigned.discard(j)
    cid += 1
nd1 = json.load(open(OUTPUT_DIR/"rag_analysis_v1/near_duplicate_summary.json"))
pd.DataFrame({
  "v1": {"clusters@0.9": nd1["n_clusters_at_0.9"], "mean_NN_cos": nd1["mean_nn_cos"],
         "pct_twins>0.9": nd1["pct_rows_with_nn_gt_0.9"], "guideline_reach_%": 6.5},
  "v2": {"clusters@0.9": cid, "mean_NN_cos": round(float(nn.mean()),4),
         "pct_twins>0.9": round(float((nn>0.9).mean()*100),2), "guideline_reach_%": 17.6}}).T

,clusters@0.9,mean_NN_cos,pct_twins>0.9,guideline_reach_%
v1,173.0,0.9356,81.66,6.5
v2,237.0,0.9149,67.59,17.6


# === Atomic-claim verification (FActScore-lite) ===

### === Results (verifier: gemma4, different family; method in scripts/v2/factscore_lite.py) ===

In [42]:
fs = json.load(open(OUTPUT_DIR/"factscore_lite.json"))
print({k: fs[k] for k in ("n_explanations","n_claims","pct_supported","pct_unsupported","pct_unverifiable")})
claims = pd.read_csv(OUTPUT_DIR/"factscore_lite_claims.csv")
print("\nrandom claims:")
for _, r in claims.sample(5, random_state=3).iterrows():
    print(f"  [{r.verdict:14s}] {r.claim[:100]}")

{'n_explanations': 60, 'n_claims': 797, 'pct_supported': 52.32, 'pct_unsupported': 0.0, 'pct_unverifiable': 47.68}

random claims:
  [SUPPORTED     ] The recommendation is to verify patient position (supine/sitting vs standing).
  [SUPPORTED     ] Recent physical activity should be considered before concluding elevated EDA indicates acute stress.
  [UNVERIFIABLE  ] The high standard deviation has a z-score of +9.2.
  [UNVERIFIABLE  ] The detected pattern suggests acute sympathetic arousal.
  [SUPPORTED     ] The anomaly suggests a sudden sympathetic stress response indicated by high electrodermal activity (


# === Ablations ===

In [43]:
for sg in ("main","wordcap","genablation"):
    sel = [r for r in rows if r["subgroup"]==sg and (sg!="main" or r["group"]=="dalia")]
    words = np.mean([len(r["explanation"].split()) for r in sel])
    s = ev[ev.subgroup==sg] if sg!="main" else ev[(ev.subgroup=="main") & (ev.group=="dalia")]
    print(f"{sg:12s} n={len(sel):3d} words={words:6.1f} faith={s.local_faithfulness.mean():.2f} compl={s.local_completeness.mean():.2f}")

main         n=398 words= 127.3 faith=2.21 compl=2.20
wordcap      n= 50 words= 196.5 faith=2.02 compl=2.12
genablation  n= 50 words=  99.1 faith=1.78 compl=1.52


# === Example alerts ===

### === One concordant explanation ===

In [44]:
ex = next(r for r in rows if r["group"]=="wesad"
          and "stress" in sec(r["explanation"],"DETECTED","EVIDENCE")
          and "artifact" not in sec(r["explanation"],"DETECTED","EVIDENCE"))
print("TRUE LABEL:", ex["true_label"], "| QUERY:", ex["query"][:180])
print()
print(ex["explanation"][:800])

TRUE LABEL: stress | QUERY: Biosignal window flagged by anomaly detection. reduced bvp (z=-7.2 vs subject baseline) elevated wrist_eda (z=+4.0 vs subject baseline). Relevant topics: electrodermal activity ski

DETECTED: The pattern suggests acute sympathetic arousal (stress) where reduced blood volume pulse and elevated skin conductance indicate a physiological stress response common in anxiety or high cognitive load scenarios .

EVIDENCE: EDA reflects changes from sweat gland activity modulated by the autonomic nervous system, making it valuable for detecting emotional arousal and stress . When stressed, blood pressure increases causing higher heart rate linked with low HRV (reduced BVP) . EDA tends to increase during stressful periods while adding noise can affect PPG signals like the reduced BVP seen here .

RECOMMENDATION: Monitor for sustained elevation; consider multimodal confirmation if clinical context warrants, as single-signal methods have limitations in real-world settings 

### === One typical pathology failure ===

In [45]:
ptb = next(r for r in rows if r["group"]=="ptbxl")
print("TRUE LABEL:", ptb["true_label"])
print("DETECTED:", sec(ptb["explanation"], "DETECTED", "EVIDENCE")[:400])

TRUE LABEL: MI
DETECTED: the high anomaly score suggests an irregular rhythm or ectopic beat pattern that deviates significantly from normal sinus morphology.


# Reading

- Validated judge (gemma4): faithfulness 2.21/3 on wearable alerts, 44% fully faithful, 2 hallucination verdicts — a distribution, not a "zero hallucination" headline.
- Raw citation accuracy 99.01% (12 fabrications in 9/546 explanations); repair drops them.
- Concordance: 94% stress / 12% ectopy / 6% pathology, with 42–56% of true pathology attributed to artifact — the safety-critical finding.
- 47.7% of atomic claims unverifiable from the retrieved context.
- v2 fixes raised guideline reach (6.5→17.6%) and cut duplication (173→237 clusters) at the cost of narrower document spread (53→44).
- API-judge columns await OpenRouter key renewal; clinician ratings via `clinician_eval/`.